# 🏛️ UIT Legal IR — 2-Stage Retrieval & Re-ranking (Chuẩn BTC SoICT/UIT)## Pipeline: BM25 + Bi-Encoder → PhoRanker Cross-Encoder Re-ranking**Kiến trúc dựa trên nghiên cứu chính thức của BTC (arXiv:2507.14619v1 — Team 4Huiter, Top 3 SoICT Hackathon)**### Datasets cần thêm vào Notebook:| Dataset | Đường dẫn Kaggle ||:---|:---|| Dữ liệu luật | `/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data` || Mô hình Fine-tuned | `/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model` || Cache PKL (4 file) | `/kaggle/input/datasets/thurdayafternoon/pkl-cache` |### Settings Notebook:- **Accelerator**: GPU T4 x2- **Internet**: ON (cần để clone GitHub và tải PhoRanker từ HuggingFace)- **Persistence**: Files

In [ ]:
import subprocess, re# Kiểm tra GPU để cài PyTorch phù hợp (P100 cần CUDA 11.x, T4 dùng CUDA 12.x)try:    gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True).strip()    print(f"🎮 GPU: {gpu_info}")    cap = float(re.search(r'(\d+\.\d+)', gpu_info.split(',')[-1]).group(1))    if cap < 7.0:        print("⚠️ GPU compute capability < 7.0 (P100). Cài PyTorch CUDA 11.8...")        !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118    else:        print("✅ GPU tương thích với PyTorch CUDA mặc định.")except Exception as e:    print(f"ℹ️ Không phát hiện GPU: {e}")!pip install -q sentence-transformers rank-bm25 pyvi scikit-learn matplotlib tqdm

In [ ]:
import osimport shutilWORK_DIR = "/kaggle/working"os.chdir(WORK_DIR)# === 1. CLONE CODE TỪ GITHUB ===REPO_URL = "https://github.com/manh123-chatgpt/implement-pp1.git"CLONE_DIR = os.path.join(WORK_DIR, "code")if os.path.exists(CLONE_DIR):    shutil.rmtree(CLONE_DIR)print("📦 Đang clone code từ GitHub...")os.system(f"git clone {REPO_URL} {CLONE_DIR}")# Copy toàn bộ file .py từ repo vào working dirfor f in os.listdir(CLONE_DIR):    if f.endswith(".py"):        src = os.path.join(CLONE_DIR, f)        dst = os.path.join(WORK_DIR, f)        shutil.copy2(src, dst)        print(f"  ✅ {f}")print(f"\n✅ Đã copy tất cả file Python vào {WORK_DIR}")# === 2. CẤU HÌNH ĐƯỜNG DẪN KAGGLE DATASETS ===KAGGLE_DATA_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-ir-data/uit-legal-ir-data"KAGGLE_MODEL_DIR = "/kaggle/input/datasets/thurdayafternoon/uit-legal-finetuned-model/fine_tuned_vietnamese_bi_encoder"KAGGLE_CACHE_DIR = "/kaggle/input/datasets/thurdayafternoon/pkl-cache"# === 3. COPY / SYMLINK DỮ LIỆU VÀO WORKING DIR ===# 3a. Copy legal_corpus_resolved.json (cần ghi được nên phải copy)RESOLVED_SRC = os.path.join(KAGGLE_DATA_DIR, "legal_corpus_resolved.json")RESOLVED_DST = os.path.join(WORK_DIR, "legal_corpus_resolved.json")if os.path.exists(RESOLVED_SRC) and not os.path.exists(RESOLVED_DST):    print("📄 Đang copy legal_corpus_resolved.json...")    shutil.copy2(RESOLVED_SRC, RESOLVED_DST)    print(f"  ✅ {os.path.getsize(RESOLVED_DST) / 1024**2:.0f} MB")# 3b. Symlink thư mục fine_tuned_vietnamese_bi_encoderMODEL_LINK = os.path.join(WORK_DIR, "fine_tuned_vietnamese_bi_encoder")if not os.path.exists(MODEL_LINK) and os.path.exists(KAGGLE_MODEL_DIR):    os.symlink(KAGGLE_MODEL_DIR, MODEL_LINK)    print(f"🔗 Symlink mô hình: {MODEL_LINK} → {KAGGLE_MODEL_DIR}")# 3c. Copy 4 file cache PKL (nếu có) PKL_FILES = [    "bm25_tokenized_cache_pyvi_v2.pkl",    "corpus_embeddings_bgem3_resolved.pkl",    "corpus_embeddings_e5_resolved.pkl",    "corpus_embeddings_finetuned_resolved.pkl",]for pkl in PKL_FILES:    src = os.path.join(KAGGLE_CACHE_DIR, pkl)    dst = os.path.join(WORK_DIR, pkl)    if os.path.exists(src) and not os.path.exists(dst):        shutil.copy2(src, dst)        size_mb = os.path.getsize(dst) / 1024**2        print(f"  ✅ {pkl} ({size_mb:.0f} MB)")print("\n" + "=" * 60)print("🎯 SETUP HOÀN TẤT! Sẵn sàng chạy pipeline.")print("=" * 60)

In [ ]:
import torchimport os# Kiểm tra GPUprint("🖥️ THÔNG TIN HỆ THỐNG:")print(f"  PyTorch: {torch.__version__}")print(f"  CUDA: {torch.cuda.is_available()}")if torch.cuda.is_available():    for i in range(torch.cuda.device_count()):        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB VRAM)")# Kiểm tra file đã sẵn sàngprint("\n📂 KIỂM TRA FILE:")check_files = {    "data_loader.py": "Code - Data Loader",    "bm25_retriever.py": "Code - BM25 Searcher",     "dense_retriever.py": "Code - Dense Retriever",    "hybrid_retriever.py": "Code - Hybrid 2-Stage Searcher",    "evaluator.py": "Code - Evaluator",    "mine_semi_hard_negatives.py": "Code - Semi-Hard Negative Mining",    "train_cross_encoder.py": "Code - Cross-Encoder Trainer",    "legal_corpus_resolved.json": "Data - Graph Resolved Corpus",    "fine_tuned_vietnamese_bi_encoder": "Model - Fine-tuned Bi-Encoder",    "bm25_tokenized_cache_pyvi_v2.pkl": "Cache - BM25 Tokenized",    "corpus_embeddings_finetuned_resolved.pkl": "Cache - Bi-Encoder Embeddings",}for fname, desc in check_files.items():    exists = "✅" if os.path.exists(fname) else "❌"    print(f"  {exists} {desc}: {fname}")

In [ ]:
from data_loader import load_corpus, load_train_data, load_test_datacorpus = load_corpus()train_data = load_train_data()test_data = load_test_data()print(f"\n📊 Corpus: {len(corpus)} văn bản pháp luật")print(f"📊 Train: {len(train_data)} câu hỏi")print(f"📊 Test: {len(test_data)} câu hỏi")

## 🔍 Stage 1 + Stage 2: Hybrid 2-Stage Retrieval & Re-ranking- **Stage 1**: BM25 (Lexical) + Fine-tuned Bi-Encoder (Semantic) → Lọc **Top 90** ứng viên- **Stage 2**: PhoRanker Cross-Encoder re-rank 90 ứng viên → Chọn **Top 5** chính xác nhất> ⚡ Chế độ `light_mode=True` chỉ dùng BM25 + Bi-Encoder (không cần BGE-M3/E5-Large), giúp chạy trong **< 15 phút**!

In [ ]:
from hybrid_retriever import HybridSearcherfrom evaluator import compute_metricsfrom tqdm import tqdm# Khởi tạo hệ thống 2-Stage chuẩn BTC# light_mode=True: Chỉ BM25 + Bi-Encoder + PhoRanker (nhanh, tiết kiệm GPU)# light_mode=False: Thêm BGE-M3 + E5-Large (chậm hơn nhưng có thể mạnh hơn nếu có cache)hybrid = HybridSearcher(corpus, use_reranker=True, light_mode=True)# --- Đánh giá trên 500 câu hỏi Validation ---val_items = list(train_data.items())[:500]val_questions = {k: v["question"] for k, v in val_items}val_truth = {k: v["answer"] for k, v in val_items}print("\n🔍 Đang đánh giá trên 500 câu hỏi validation...")val_preds = {}for qid, question in tqdm(val_questions.items(), desc="2-Stage Validation"):    val_preds[qid] = hybrid.search(question, top_k=5)results = compute_metrics(val_preds, val_truth, k=5)print("\n" + "=" * 50)print(f"📊 KẾT QUẢ 2-STAGE HYBRID (BM25 + Bi-Encoder + PhoRanker):")print(f"👉 Recall@5   : {results['Recall'] * 100:.2f}%")print(f"👉 Precision@5: {results['Precision'] * 100:.2f}%")print("=" * 50)

## 📦 Tạo file Submission cho Public Test (999 câu hỏi)

In [ ]:
import jsonimport zipfile# Dự đoán trên toàn bộ 999 câu hỏi Public Testprint("📦 Đang sinh kết quả cho 999 câu hỏi Public Test...")test_preds = {}for qid, item in tqdm(test_data.items(), desc="Predicting Public Test"):    question = item["question"]    test_preds[qid] = hybrid.search(question, top_k=5)# Tạo file submission chuẩn BTCformatted_preds = {}for qid, docs in test_preds.items():    formatted_preds[str(qid)] = {        "answer": [str(d) for d in docs[:5]]    }with open("submission.json", "w", encoding="utf-8") as f:    json.dump(formatted_preds, f, ensure_ascii=False, indent=4)with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:    zf.write("submission.json", arcname="submission.json")print(f"\n🎉 Đã tạo thành công file nộp bài: 'submission.zip'")print(f"📊 Tổng số câu hỏi đã dự đoán: {len(formatted_preds)}")# Xem thử 3 câu đầu tiênfor i, (qid, pred) in enumerate(list(formatted_preds.items())[:3]):    q_text = test_data[qid]["question"][:80]    print(f"\n  [{qid}] {q_text}...")    print(f"  → Đáp án: {pred['answer']}")

## 📊 (Tùy chọn) Trực quan hóa Embedding Space bằng t-SNE> Bỏ qua cell này nếu chỉ cần tạo submission.

In [ ]:
from visualize_tsne import run_tsne_visualizationrun_tsne_visualization(    model_path="fine_tuned_vietnamese_bi_encoder",    num_samples=200,    output_image="embedding_tsne_visualization.png")from IPython.display import Image, displaydisplay(Image("embedding_tsne_visualization.png", width=900))

## 🔧 (Tùy chọn) Huấn luyện nâng cao: Semi-Hard Negative Mining + Cross-Encoder> ⚠️ Chỉ chạy nếu muốn huấn luyện lại Cross-Encoder từ đầu. Mỗi bước mất ~15-30 phút GPU.**Bước 1**: Khai thác mẫu âm bán khó (Semi-Hard Negatives) từ Top 90 Bi-Encoder  **Bước 2**: Huấn luyện PhoRanker Cross-Encoder với BCEWithLogitsLoss (2 epochs)

In [ ]:
# === Bước 1: Khai thác Semi-Hard Negatives ===from mine_semi_hard_negatives import mine_semi_hard_negativesmine_semi_hard_negatives()

In [ ]:
# === Bước 2: Huấn luyện Cross-Encoder (PhoRanker) ===from train_cross_encoder import train_cross_encodertrain_cross_encoder()